# DWS-Bench: Kaggle benchmark generation and evaluation

This notebook is configured for a Kaggle GPU notebook. It:

1. Locates the uploaded repository under `/kaggle/input` or clones it from GitHub.
2. Installs the project dependencies.
3. Generates the current reduced benchmark (50 instances per condition, about 1,150 records).
4. Evaluates the five core models in separate processes with auditable step prompts.
5. Creates per-query JSONL and CSV audits, metrics, Markdown reports, plots, and one ZIP archive.

The Hugging Face token must be available as the Kaggle secret/environment variable `HF_TOKEN` for gated models such as Llama.

In [ ]:
# Install Kaggle-side helpers before locating the uploaded project.
%pip install -q bitsandbytes pandas matplotlib
print("Kaggle helper packages installed.")

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

# Kaggle datasets are mounted read-only under /kaggle/input.
# Upload the repository as a Kaggle Dataset, or set REPO_URL to a public/private clone URL.
REPO_URL = ""
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
PROJECT_DIR = WORK_ROOT / "StateMachine"

if REPO_URL:
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    candidates = [
        path for path in KAGGLE_INPUT_ROOT.rglob("run_eval.py")
        if (path.parent / "generate_all.py").exists()
    ]
    if not candidates:
        raise FileNotFoundError(
            "Upload the StateMachine repository as a Kaggle Dataset, or set REPO_URL."
        )
    source_dir = candidates[0].parent
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    shutil.copytree(source_dir, PROJECT_DIR, ignore=shutil.ignore_patterns(
        ".git", "__pycache__", "*.pyc", "data/*.jsonl", "results"
    ))

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print(f"Project: {PROJECT_DIR}")
print(f"Source: {source_dir if not REPO_URL else REPO_URL}")

In [ ]:
# Install the exact project dependencies after the repository is available.
%pip install -q -r requirements.txt bitsandbytes pandas matplotlib

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA capability:", torch.cuda.get_device_capability(0))
else:
    raise RuntimeError("A Kaggle GPU runtime is required for the full model evaluation.")

## Kaggle configuration

Keep the repository under `/kaggle/working` because `/kaggle/input` is read-only. Set the `HF_TOKEN` Kaggle secret before running the model cells if you will evaluate Llama. The notebook requests explicit `Step N: <container>` lines, so step-wise scoring measures auditable state answers rather than hidden chain-of-thought.

In [ ]:
import os
from pathlib import Path

# Prefer Kaggle Secrets Manager; never print the token.
if not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        secret_token = UserSecretsClient().get_secret("HF_TOKEN")
        if secret_token:
            os.environ["HF_TOKEN"] = secret_token
    except Exception:
        pass

DATASET = "full"
PRECISION = "4bit"       # Suitable starting point for Kaggle T4/P100 GPUs.
BATCH_SIZE = 1            # Increase only after checking GPU memory.
DEVICE = "auto"
OUTPUT_ROOT = Path("/kaggle/working/dws_controlled_reruns")
MODELS = ["qwen2.5-0.5b", "qwen2.5-3b", "qwen2.5-7b"]
TOKEN_BUDGETS = [128, 256, 512]
CONDITIONS = [
    {"model": model, "cot": cot, "max_new_tokens": tokens}
    for model in MODELS
    for cot in (False, True)
    for tokens in TOKEN_BUDGETS
]

if not os.environ.get("HF_TOKEN"):
    print("HF_TOKEN is not set. Public Qwen models can run; gated models are excluded.")
else:
    print("HF_TOKEN detected without printing its value.")

print("Conditions:", len(CONDITIONS))
print("Output root:", OUTPUT_ROOT)
for condition in CONDITIONS:
    print(condition)

In [ ]:
# The old generated JSONL files were removed; regenerate the current reduced benchmark.
dataset_path = PROJECT_DIR / "data" / "full_benchmark.jsonl"
if not dataset_path.exists():
    subprocess.run([sys.executable, "generate_all.py"], cwd=PROJECT_DIR, check=True)
else:
    print(f"Using existing dataset: {dataset_path}")

records = [line for line in dataset_path.open(encoding="utf-8") if line.strip()]
record_count = len(records)
print(f"Full benchmark records: {record_count}")
if record_count < 1000 or record_count > 1200:
    raise ValueError(f"Expected the reduced benchmark to contain about 1,150 records; found {record_count}.")

## Run all core models

Each model is launched in a separate subprocess. This releases model weights before the next model and allows the notebook to continue if one model fails. Each run writes predictions JSONL, an audit CSV containing prompt/response/expected answer/step comparison, metrics JSON, and a Markdown report.

In [ ]:
import subprocess
import sys
import time
import pandas as pd

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
run_status = []
for index, condition in enumerate(CONDITIONS, start=1):
    model_name = condition["model"]
    cot_label = "cot" if condition["cot"] else "no_cot"
    condition_name = f"{model_name}_{cot_label}_{condition['max_new_tokens']}"
    condition_dir = OUTPUT_ROOT / condition_name
    command = [
        sys.executable, "run_eval.py",
        "--model", model_name,
        "--dataset", DATASET,
        "--device", DEVICE,
        "--precision", PRECISION,
        "--batch-size", str(BATCH_SIZE),
        "--max-new-tokens", str(condition["max_new_tokens"]),
        "--output-dir", str(condition_dir),
    ]
    if condition["cot"]:
        command.append("--cot")

    print(f"\n[{index}/{len(CONDITIONS)}] {condition_name}")
    started = time.time()
    completed = subprocess.run(command, cwd=PROJECT_DIR, env=os.environ.copy())
    run_status.append({
        **condition,
        "condition": condition_name,
        "return_code": completed.returncode,
        "elapsed_minutes": round((time.time() - started) / 60, 2),
        "output_dir": str(condition_dir),
    })
    if completed.returncode != 0:
        print(f"FAILED: {condition_name}; continuing with remaining conditions.")

status_df = pd.DataFrame(run_status)
display(status_df)
status_df.to_csv(OUTPUT_ROOT / "run_status.csv", index=False)
print(f"Saved run status to {OUTPUT_ROOT / 'run_status.csv'}")

In [ ]:
# Aggregate only corrected prediction artifacts from this controlled rerun.
import json
from statistics import median

prediction_rows = []
condition_rows = []
for condition in CONDITIONS:
    model_name = condition["model"]
    cot_label = "cot" if condition["cot"] else "no_cot"
    condition_name = f"{model_name}_{cot_label}_{condition['max_new_tokens']}"
    prediction_files = list((OUTPUT_ROOT / condition_name / model_name).glob("*_predictions.jsonl"))
    if not prediction_files:
        print(f"Missing predictions under: {OUTPUT_ROOT / condition_name / model_name}")
        continue

    prediction_file = prediction_files[0]
    rows = [json.loads(line) for line in prediction_file.open(encoding="utf-8") if line.strip()]
    for row in rows:
        measured = row.get("measured_factors", {})
        requested = row.get("requested_factors", {})
        row.update({
            "model": model_name,
            "cot": condition["cot"],
            "max_new_tokens": condition["max_new_tokens"],
            "condition": condition_name,
            "depth": measured.get("T_actual", requested.get("T")),
            "invalid_final_answer": not bool(row.get("has_final_answer", False)),
            "truncated": row.get("finish_reason") == "length",
        })
        prediction_rows.append(row)

    token_values = [row["generated_tokens"] for row in rows if row.get("generated_tokens") is not None]
    total = len(rows)
    condition_rows.append({
        "model": model_name,
        "cot": condition["cot"],
        "max_new_tokens": condition["max_new_tokens"],
        "condition": condition_name,
        "instances": total,
        "accuracy": sum(bool(row.get("is_correct")) for row in rows) / total if total else 0.0,
        "invalid_rate": sum(not bool(row.get("has_final_answer")) for row in rows) / total if total else 0.0,
        "truncation_rate": sum(row.get("finish_reason") == "length" for row in rows) / total if total else 0.0,
        "mean_generated_tokens": sum(token_values) / len(token_values) if token_values else None,
        "median_generated_tokens": median(token_values) if token_values else None,
    })

summary_df = pd.DataFrame(condition_rows).sort_values(["model", "cot", "max_new_tokens"])
families_df = (
    pd.DataFrame(prediction_rows)
    .groupby(["model", "cot", "max_new_tokens", "family"], dropna=False)
    .agg(instances=("is_correct", "size"), accuracy=("is_correct", "mean"), invalid_rate=("invalid_final_answer", "mean"), truncation_rate=("truncated", "mean"))
    .reset_index()
)
depth_df = (
    pd.DataFrame(prediction_rows)
    .groupby(["model", "cot", "max_new_tokens", "depth"], dropna=False)
    .agg(instances=("is_correct", "size"), accuracy=("is_correct", "mean"), invalid_rate=("invalid_final_answer", "mean"), truncation_rate=("truncated", "mean"))
    .reset_index()
)

# Positive values mean CoT improved accuracy over No-CoT at the same token budget.
comparison_df = summary_df.pivot_table(
    index=["model", "max_new_tokens"], columns="cot", values="accuracy"
).reset_index()
comparison_df = comparison_df.rename(columns={False: "no_cot_accuracy", True: "cot_accuracy"})
comparison_df["cot_delta"] = comparison_df["cot_accuracy"] - comparison_df["no_cot_accuracy"]

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(OUTPUT_ROOT / "corrected_condition_summary.csv", index=False)
families_df.to_csv(OUTPUT_ROOT / "corrected_family_accuracy.csv", index=False)
depth_df.to_csv(OUTPUT_ROOT / "corrected_depth_accuracy.csv", index=False)
comparison_df.to_csv(OUTPUT_ROOT / "cot_vs_no_cot.csv", index=False)

display(summary_df.style.format({
    "accuracy": "{:.2%}", "invalid_rate": "{:.2%}",
    "truncation_rate": "{:.2%}", "mean_generated_tokens": "{:.1f}",
    "median_generated_tokens": "{:.1f}",
}))
print(f"Analyzed {len(prediction_rows)} corrected predictions.")
print(f"Saved summaries under {OUTPUT_ROOT}")

In [ ]:
# Package the corrected matrix artifacts for download.
archive_path = shutil.make_archive(
    str(WORK_ROOT / "dws_bench_corrected_matrix"),
    "zip",
    root_dir=OUTPUT_ROOT.parent,
    base_dir=OUTPUT_ROOT.name,
)
print(f"Created archive: {archive_path}")
print("Kaggle output directory:", OUTPUT_ROOT)
print("Download the ZIP from the notebook output/files panel after the run completes.")